# 02b — Pre-processing: *di mana* & *mengapa*
## Construction Safety Helmet — YOLO26

> ⚠️ **Notebook ini TIDAK wajib dijalankan sebelum training.**
> Isinya murni **penjelasan & visualisasi** untuk *memahami* pre-processing. Pre-processing yang sebenarnya (letterbox + normalisasi + augmentasi) terjadi **otomatis di dalam `src/train.py`** saat training, jadi kamu boleh langsung ke `03_train` tanpa menjalankan notebook ini.

Dataset ini **mentah** (Roboflow tidak menerapkan apa pun), dan pre-processing dilakukan **otomatis oleh Ultralytics saat runtime** — bukan sebagai script tersendiri. Di bawah dijelaskan + divisualisasikan apa yang sebenarnya terjadi pada tiap gambar sebelum masuk ke model.

## Setup

In [ ]:
import os, sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "src").is_dir() and (d / "dataset").exists():
            return d
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import eda
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
from PIL import Image
print("PROJECT_ROOT:", PROJECT_ROOT)

# === Auto-simpan grafik ke presentation_figures/ (otomatis tiap kali di-Run) ===
import matplotlib.pyplot as plt
_FIGDIR = PROJECT_ROOT / "presentation_figures"; _FIGDIR.mkdir(exist_ok=True)
_NB = "02b_preprocessing"; _cnt = {"n": 0}
for _old in _FIGDIR.glob(_NB + "_*.png"):
    try: _old.unlink()
    except Exception: pass
_orig_show = getattr(plt, "_orig_show", plt.show); plt._orig_show = _orig_show
def _autosave_show(*a, **k):
    _f = plt.gcf()
    if _f.get_axes():
        _cnt["n"] += 1
        try: _f.savefig(_FIGDIR / (_NB + "_%02d.png" % _cnt["n"]), dpi=130, bbox_inches="tight")
        except Exception: pass
    return _orig_show(*a, **k)
plt.show = _autosave_show

## 1. Fakta: dataset ini **mentah**
Roboflow mencatat sendiri bahwa tidak ada pra-pemrosesan/augmentasi pada export ini. Akibatnya gambar berukuran **beragam** (bukan 640×640), persis seperti temuan EDA §8.

In [ ]:
# Kutip baris relevan dari README export Roboflow
readme = Path("dataset/README.roboflow.txt").read_text(encoding="utf-8", errors="replace")
for line in readme.splitlines():
    if "pre-processing" in line.lower() or "augmentation" in line.lower():
        print("README Roboflow:", line.strip())

# Bukti ukuran gambar beragam (beberapa sampel)
data, class_names, nc, root = eda.load_dataset_config("dataset/data.yaml")
vimg, vlbl = eda.resolve_splits(root, data)["valid"]
sample = sorted(p for p in vimg.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})[:6]
print("\nContoh ukuran gambar asli (W x H):")
for p in sample:
    with Image.open(p) as im:
        print(f"  {p.name[:40]:40} {im.size[0]} x {im.size[1]}")

## 2. Tiga langkah pre-processing (otomatis, saat runtime)
Setiap gambar melewati ini **setiap kali dimuat** oleh Ultralytics (train/val/infer):

| # | Langkah | Apa yang terjadi | Dikontrol oleh | Kapan |
|---|---|---|---|---|
| 1 | **Letterbox 640** | resize menjaga rasio + padding abu-abu (114) ke 640×640 | `imgsz` (`config.yaml`) | train, val, infer |
| 2 | **Normalisasi** | piksel `0–255 → 0–1` (tanpa mean/std ala ImageNet) | otomatis | train, val, infer |
| 3 | **Augmentasi** | mosaic, jitter HSV, flip, scale, translate, … | default Ultralytics | **hanya training** |

Karena ini *runtime*, tidak ada gambar baru yang ditulis ke disk — dataset tetap **read-only**. Membuat `preprocess.py` manual (mis. menyimpan ulang semua gambar 640×640) justru **redundan & berisiko** (objek kecil rusak, dan augmentasi tetap harus di-runtime).

## 3. Visual — Langkah 1: Letterbox 640 (*sebelum → sesudah*)
Inilah yang membuat gambar beragam ukuran menjadi 640×640 seragam tanpa mengubah rasio objek. Area abu-abu = **padding** (kenapa gambar non-1:1 'boros' sebagian kanvas).

In [ ]:
def letterbox(im, new=640, color=(114, 114, 114)):
    W, H = im.size
    r = min(new / W, new / H)                      # skala menjaga rasio
    nw, nh = round(W * r), round(H * r)
    canvas = Image.new("RGB", (new, new), color)   # kanvas abu-abu 114 (khas YOLO)
    px, py = (new - nw) // 2, (new - nh) // 2       # padding kiri/atas
    canvas.paste(im.convert("RGB").resize((nw, nh)), (px, py))
    return canvas, r, px, py

# pilih satu gambar val non-persegi yang punya anotasi
def gt_boxes(stem):
    f = vlbl / (stem + ".txt"); out = []
    if f.exists():
        for ln in f.read_text().splitlines():
            q = ln.split()
            if len(q) == 5:
                try: out.append((int(q[0]), *map(float, q[1:])))
                except ValueError: pass
    return out
pick = None
for p in sample + sorted(vimg.iterdir()):
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
    with Image.open(p) as im: W, H = im.size
    if abs(W - H) > 0.15 * max(W, H) and gt_boxes(p.stem):
        pick = p; break
pick = pick or sample[0]

with Image.open(pick) as im:
    orig = im.convert("RGB"); W, H = orig.size
boxes = gt_boxes(pick.stem)
CLASS_COLORS = {0: "#4c72b0", 1: "#c44e52", 2: "#55a868"}
lb, r, px, py = letterbox(orig, 640)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(orig)
for c, x, y, w, h in boxes:
    ax[0].add_patch(mpatches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H, fill=False, edgecolor=CLASS_COLORS.get(c,"y"), lw=2))
ax[0].set_title(f"SEBELUM — asli {W}x{H}"); ax[0].axis("off")
ax[1].imshow(lb)
for c, x, y, w, h in boxes:                         # transformasi kotak: skala r + offset pad
    nx, ny, nw_, nh_ = x*W*r + px, y*H*r + py, w*W*r, h*H*r
    ax[1].add_patch(mpatches.Rectangle((nx-nw_/2, ny-nh_/2), nw_, nh_, fill=False, edgecolor=CLASS_COLORS.get(c,"y"), lw=2))
ax[1].add_patch(mpatches.Rectangle((px, py), round(W*r), round(H*r), fill=False, edgecolor="white", ls="--", lw=1))
ax[1].set_title(f"SESUDAH — letterbox 640x640 (skala={r:.2f}, padding abu-abu)"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print(f"=> Rasio objek TIDAK berubah; gambar diskalakan {r:.2f}x lalu di-pad. "
      f"Padding: kiri/kanan={px}px, atas/bawah={py}px.")

## 4. Visual — Langkah 3: Augmentasi training yang **benar-benar dipakai**
Saat training, Ultralytics mencatat nilai augmentasi ke `args.yaml`. Di bawah nilai **asli** yang dipakai (dari run terakhir) + artinya. Lalu `train_batch0.jpg` adalah **batch nyata setelah pre-processing + augmentasi** yang masuk ke model.

In [4]:
import yaml
EXPLAIN = {
    "imgsz": "ukuran input setelah letterbox",
    "hsv_h": "jitter HUE (warna)", "hsv_s": "jitter SATURASI", "hsv_v": "jitter VALUE (kecerahan)",
    "degrees": "rotasi (derajat)", "translate": "geser posisi", "scale": "zoom in/out",
    "shear": "miringkan", "perspective": "distorsi perspektif",
    "flipud": "flip vertikal (prob)", "fliplr": "flip horizontal (prob)",
    "mosaic": "gabung 4 gambar jadi 1 (prob)", "mixup": "campur 2 gambar (prob)",
    "copy_paste": "tempel objek antar-gambar (prob)", "erasing": "hapus area acak (prob)",
}
args_path = Path("runs/train/helmet_yolo26s_baseline/args.yaml")
if args_path.exists():
    args = yaml.safe_load(args_path.read_text(encoding="utf-8"))
    rows = [(k, args.get(k), EXPLAIN[k]) for k in EXPLAIN if k in args]
    display(pd.DataFrame(rows, columns=["parameter", "nilai dipakai", "arti"]).set_index("parameter"))
    print("Nilai 0 / 0.0 = augmentasi itu nonaktif. train.py tidak meng-override => default Ultralytics.")
else:
    print("args.yaml belum ada — jalankan training (notebook 03) untuk melihat nilai augmentasi nyata.")
    print("Default Ultralytics yang umum: mosaic=1.0, fliplr=0.5, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1.")

Dan inilah **batch training nyata** (sudah letterbox + augmentasi mosaic) yang masuk ke model:

In [5]:
from IPython.display import Image as IPImage, display
batch = Path("runs/train/helmet_yolo26s_baseline/train_batch0.jpg")
if batch.exists():
    print("Batch training nyata (sudah letterbox + augmentasi mosaic):")
    display(IPImage(filename=str(batch)))
    print("=> Inilah wujud data yang benar-benar 'dilihat' model — bukan gambar asli.")
else:
    print("train_batch0.jpg belum ada — jalankan training (notebook 03) dulu.")

## 5. Ringkasan — *kenapa* tak ada `preprocess.py`

| Pertanyaan | Jawaban |
|---|---|
| Apakah ada pre-processing? | **Ada**, tapi otomatis di runtime Ultralytics (letterbox + normalisasi + augmentasi). |
| Kenapa bukan script terpisah? | Dataset **read-only** (export verbatim) & Ultralytics mengharapkan input **mentah** lalu memproses sendiri. |
| Di mana saya mengontrolnya? | `imgsz` di `config.yaml` (resize) + hyperparameter augmentasi (default Ultralytics; bisa di-override di `train.py`). |
| Di mana saya melihat hasilnya? | Letterbox → §3 di sini; augmentasi nyata → `train_batch0.jpg` (§4) & preview di `03_train`. |

**Intinya:** pre-processing menyatu di langkah training, bukan tahap manual — membuatnya terpisah hanya akan menduplikasi pekerjaan Ultralytics dan menyentuh dataset yang seharusnya tak diubah.